<a href="https://colab.research.google.com/github/Rafa-Cami/CD-Norton/blob/main/EP3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

vamos começar LET'S GO BITCHESSSSS

# Imports

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

# Seed

Seed definida para garantir reprodutibilidade

In [21]:
seed = 42

random.seed(seed)
np.random.seed(seed)

### Importando o conjunto de dados 

In [10]:
df = pd.read_csv('subamostra_90_porcento.csv')
df.head(50)

,age,gender,marital_status,education_level,employment_status,sleep_hours,physical_activity_hours_per_week,screen_time_hours_per_day,social_support_score,work_stress_level,...,depression_score,stress_level,mood_swings_frequency,concentration_difficulty_level,panic_attack_history,family_history_mental_illness,previous_mental_health_diagnosis,therapy_history,substance_use,mental_health_risk
0,47,Female,Single,High School,Unemployed,5.3,1.5,7.2,8,5,...,5,4,6,3,1,1,1,1,0,1
1,53,Other,Single,Bachelor,Employed,4.8,0.5,5.6,8,4,...,7,1,2,5,1,1,0,0,0,2
2,43,Male,Single,Bachelor,Student,6.4,6.9,10.2,4,1,...,8,9,9,5,1,0,1,1,0,1
3,18,Other,Married,High School,Unemployed,9.6,0.4,7.4,6,6,...,7,5,7,3,0,0,1,0,1,0
4,23,Female,Divorced,Master,Self-Employed,9.0,1.3,5.1,6,3,...,4,5,7,7,0,1,0,0,0,0
5,53,Female,Married,High School,Unemployed,7.9,14.3,6.9,8,5,...,3,6,8,10,0,0,0,0,1,0
6,45,Other,Married,Master,Student,8.2,2.0,3.0,2,8,...,2,7,1,3,0,1,0,1,1,1
7,23,Other,Divorced,Master,Employed,7.8,4.7,9.1,7,6,...,6,10,3,7,0,0,1,1,1,0
8,42,Other,Single,Bachelor,Student,9.5,9.0,9.8,7,2,...,8,3,6,1,1,1,1,1,0,1
9,34,Female,Divorced,PhD,Employed,6.7,5.2,2.6,7,8,...,8,8,10,6,1,1,0,1,1,2


# Funções de plots das métricas

# Tratamento da base

### Tratamento das variáveis categóricas texto para numéricas

Mapeamento manual Ordinal encoding para não perder a ordem lógica da variável `education_level`

In [13]:
# Defina a ordem correta baseada nos dados que você tem
mapa_educacao = {
    'High School': 1,
    'Bachelor': 2,
    'Master': 3,
    'PhD': 4  # adicione outras categorias se houver
}

df_edu = df.copy()

df_edu['education_level'] = df_edu['education_level'].map(mapa_educacao)
df_edu.head(10)


,age,gender,marital_status,education_level,employment_status,sleep_hours,physical_activity_hours_per_week,screen_time_hours_per_day,social_support_score,work_stress_level,...,depression_score,stress_level,mood_swings_frequency,concentration_difficulty_level,panic_attack_history,family_history_mental_illness,previous_mental_health_diagnosis,therapy_history,substance_use,mental_health_risk
0,47,Female,Single,1,Unemployed,5.3,1.5,7.2,8,5,...,5,4,6,3,1,1,1,1,0,1
1,53,Other,Single,2,Employed,4.8,0.5,5.6,8,4,...,7,1,2,5,1,1,0,0,0,2
2,43,Male,Single,2,Student,6.4,6.9,10.2,4,1,...,8,9,9,5,1,0,1,1,0,1
3,18,Other,Married,1,Unemployed,9.6,0.4,7.4,6,6,...,7,5,7,3,0,0,1,0,1,0
4,23,Female,Divorced,3,Self-Employed,9.0,1.3,5.1,6,3,...,4,5,7,7,0,1,0,0,0,0
5,53,Female,Married,1,Unemployed,7.9,14.3,6.9,8,5,...,3,6,8,10,0,0,0,0,1,0
6,45,Other,Married,3,Student,8.2,2.0,3.0,2,8,...,2,7,1,3,0,1,0,1,1,1
7,23,Other,Divorced,3,Employed,7.8,4.7,9.1,7,6,...,6,10,3,7,0,0,1,1,1,0
8,42,Other,Single,2,Student,9.5,9.0,9.8,7,2,...,8,3,6,1,1,1,1,1,0,1
9,34,Female,Divorced,4,Employed,6.7,5.2,2.6,7,8,...,8,8,10,6,1,1,0,1,1,2


One-hot encoding para as variáveis `gender`, `marital_status` e `employment_status`

In [14]:
df_tratado = pd.get_dummies(df_edu, columns=['gender', 'marital_status', 'employment_status'], dtype=int)
df_tratado.head(10)

,age,education_level,sleep_hours,physical_activity_hours_per_week,screen_time_hours_per_day,social_support_score,work_stress_level,academic_pressure_level,job_satisfaction_score,financial_stress_level,...,gender_Female,gender_Male,gender_Other,marital_status_Divorced,marital_status_Married,marital_status_Single,employment_status_Employed,employment_status_Self-Employed,employment_status_Student,employment_status_Unemployed
0,47,1,5.3,1.5,7.2,8,5,8,10,10,...,1,0,0,0,0,1,0,0,0,1
1,53,2,4.8,0.5,5.6,8,4,7,7,10,...,0,0,1,0,0,1,1,0,0,0
2,43,2,6.4,6.9,10.2,4,1,5,8,4,...,0,1,0,0,0,1,0,0,1,0
3,18,1,9.6,0.4,7.4,6,6,9,8,5,...,0,0,1,0,1,0,0,0,0,1
4,23,3,9.0,1.3,5.1,6,3,9,10,7,...,1,0,0,1,0,0,0,1,0,0
5,53,1,7.9,14.3,6.9,8,5,3,8,4,...,1,0,0,0,1,0,0,0,0,1
6,45,3,8.2,2.0,3.0,2,8,2,1,10,...,0,0,1,0,1,0,0,0,1,0
7,23,3,7.8,4.7,9.1,7,6,9,4,10,...,0,0,1,1,0,0,1,0,0,0
8,42,2,9.5,9.0,9.8,7,2,6,8,4,...,0,0,1,0,0,1,0,0,1,0
9,34,4,6.7,5.2,2.6,7,8,7,5,10,...,1,0,0,1,0,0,1,0,0,0


### Verificando a proproção da variavel alvo no conjunto de dados

In [22]:
contagem_risco = df['mental_health_risk'].value_counts()

print(contagem_risco)

mental_health_risk
1    10667
0     8392
2     3441
Name: count, dtype: int64


### Mapeamento das variáveis para os fatores

In [15]:
mapeamento_fatores = {
    # --- Estilo de Vida ---
    'sleep_hours': 'Estilo de Vida',
    'physical_activity_hours_per_week': 'Estilo de Vida',
    'screen_time_hours_per_day': 'Estilo de Vida',
    'substance_use': 'Estilo de Vida',
    
    # --- Psicológico e Emocional ---
    'depression_score': 'Psicológico e Emocional',
    'stress_level': 'Psicológico e Emocional',
    'mood_swings_frequency': 'Psicológico e Emocional',
    'concentration_difficulty_level': 'Psicológico e Emocional',
    
    # --- Socioeconômico e Ocupacional ---
    'age': 'Socioeconômico e Ocupacional',
    'education_level': 'Socioeconômico e Ocupacional',
    'work_stress_level': 'Socioeconômico e Ocupacional',
    'employment_status_Employed': 'Socioeconômico e Ocupacional',
    'employment_status_Self-Employed': 'Socioeconômico e Ocupacional',
    'employment_status_Student': 'Socioeconômico e Ocupacional',
    'employment_status_Unemployed': 'Socioeconômico e Ocupacional',
    
    # --- Suporte Social e Contexto Interpessoal ---
    'social_support_score': 'Suporte Social e Contexto Interpessoal',
    'gender_Female': 'Suporte Social e Contexto Interpessoal',
    'gender_Male': 'Suporte Social e Contexto Interpessoal',
    'gender_Other': 'Suporte Social e Contexto Interpessoal',
    'marital_status_Divorced': 'Suporte Social e Contexto Interpessoal',
    'marital_status_Married': 'Suporte Social e Contexto Interpessoal',
    'marital_status_Single': 'Suporte Social e Contexto Interpessoal',
    
    # --- Histórico Clínico e Comportamental ---
    'panic_attack_history': 'Histórico Clínico e Comportamental',
    'family_history_mental_illness': 'Histórico Clínico e Comportamental',
    'previous_mental_health_diagnosis': 'Histórico Clínico e Comportamental',
    'therapy_history': 'Histórico Clínico e Comportamental'
}

Um pouco desproporcional, cenário onde seria melhor fazer um crozz validation estratificado

# XGBoost??

# Random Forest

# Comparação entre os modelos